In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from prophet import Prophet

# ==========================================
# 1. DATA IMPORTATION & MOCK DATA GENERATION
# ==========================================
def load_data(file_path=None):
    """
    Imports data from CSV, Excel, or generates a mock dataset for testing.
    """
    if file_path and os.path.exists(file_path):
        print(f"--- Loading data from {file_path} ---")
        if file_path.endswith('.csv'):
            return pd.read_csv(file_path)
        elif file_path.endswith(('.xls', '.xlsx')):
            return pd.read_excel(file_path)
    
    # Fallback: Generate a synthetic time-series dataset (e.g., Daily Sales)
    print("--- No file provided or found. Generating synthetic historical data... ---")
    np.random.seed(42)
    dates = pd.date_range(start="2023-01-01", end="2025-12-31", freq="D")
    
    # Create a trend + seasonality + noise pattern
    trend = np.linspace(100, 500, len(dates))
    seasonality = 50 * np.sin(2 * np.pi * dates.dayofyear / 365.25)
    noise = np.random.normal(0, 15, len(dates))
    sales = trend + seasonality + noise
    
    # Introduce some missing values and outliers to simulate real-world data
    df = pd.DataFrame({"Date": dates, "Revenue": sales})
    df.loc[df.sample(frac=0.02).index, "Revenue"] = np.nan  # 2% missing data
    return df

# Change this path to your local file (e.g., 'sales_data.csv') to use your own data
df_raw = load_data(file_path=None)

# ==========================================
# 2. DATA CLEANING & PREPROCESSING
# ==========================================
print("\n--- Preprocessing and Cleaning Data ---")

# Step 2.1: Handle column names (Prophet requires specific columns: 'ds' and 'y')
# Mapping our columns to 'ds' (datestamp) and 'y' (target value)
df_clean = df_raw.copy()
df_clean.columns = ['ds', 'y'] 

# Step 2.2: Ensure correct data types
df_clean['ds'] = pd.to_datetime(df_clean['ds'])
df_clean['y'] = pd.to_numeric(df_clean['y'], errors='coerce')

# Step 2.3: Handle missing values (Imputation via linear interpolation)
missing_count = df_clean['y'].isnull().sum()
if missing_count > 0:
    print(f"Found {missing_count} missing values. Filling via interpolation...")
    df_clean['y'] = df_clean['y'].interpolate(method='linear')

# Step 2.4: Train-Test Split (Hold out the last 90 days for evaluation)
split_date = df_clean['ds'].max() - pd.Timedelta(days=90)
train_df = df_clean[df_clean['ds'] <= split_date]
test_df = df_clean[df_clean['ds'] > split_date]

print(f"Training data shape: {train_df.shape}")
print(f"Testing data shape: {test_df.shape}")

# ==========================================
# 3. MODEL TRAINING (PROPHET TIME-SERIES)
# ==========================================
print("\n--- Training Predictive Model ---")
# Initialize Prophet with daily and yearly seasonality enabled
model = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)
model.fit(train_df)

# ==========================================
# 4. FORECASTING FUTURE TRENDS
# ==========================================
print("\n--- Generating Forecast ---")
# Create a dataframe stretching 90 days into the test period + 180 days into the future
future_periods = len(test_df) + 180 
future = model.make_future_dataframe(periods=future_periods, freq='D')
forecast = model.predict(future)

# ==========================================
# 5. EVALUATION ACCURACY
# ==========================================
print("\n--- Evaluating Model Accuracy ---")
# Merge the predictions with actual test values to calculate metrics
performance_df = test_df.merge(forecast[['ds', 'yhat']], on='ds', how='inner')

y_true = performance_df['y']
y_pred = performance_df['yhat']

mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared Score (R²): {r2:.2f}")

# ==========================================
# 6. VISUALIZATION
# ==========================================
print("\n--- Plotting Results ---")
plt.figure(figsize=(14, 7))

# Plot historical training data
plt.plot(train_df['ds'], train_df['y'], label='Historical Train Data', color='black', alpha=0.6)

# Plot actual test data
plt.plot(test_df['ds'], test_df['y'], label='Actual Test Data (Holdout)', color='green', alpha=0.8)

# Plot predicted values
plt.plot(forecast['ds'], forecast['yhat'], label='Forecasted Trend', color='blue', linestyle='--')

# Plot uncertainty intervals
plt.fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'], 
                 color='blue', alpha=0.15, label='Confidence Interval')

# Layout formatting
plt.title('Predictive Modeling & Future Trend Forecasting', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.axvline(x=split_date, color='red', linestyle=':', label='Train/Test Split Threshold')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)

# Display the plot
plt.tight_layout()
plt.show()